UniGuide_AI
Joud Saeed Alghamdi

In [26]:
!pip install -q \
langchain \
langgraph \
langchain-google-genai \
langchain-chroma \
chromadb \
langsmith

In [27]:
import os

os.environ["GOOGLE_API_KEY"] = ""

In [28]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash"
)

response = llm.invoke(
    "Reply with exactly: UniGuide AI is ready."
)

print(response.content)

[{'type': 'text', 'text': 'UniGuide AI is ready.', 'extras': {'signature': 'Ev8DCvwDARFNMg9zfxGQ6kr438mVtrfWbQXX/OV8PCFcviOeq8Rm37c+OpuSGzeeXxcbQYsg1Yzt5YoAUaBCpUKTpHhQFQgtkFT3tu01KDAOuixYCRfJb+kZOvm2hUsZ2PTatP0EouTpygL1YjgIUj7unISopA9sLASvYFZUqFL60/YJ1tl4haKX1gsa7a6hPSjVoU4s4n2ft+8KwVmsuvqZjYiNhdJaCJU1WSLy/y+BRreoTPbrEGOexJc3u9+CeFnGZNih6MBVeKarFMNo65GXxslp1XjGbuJoJh4MrVGc9JksCx+joxC8MV8qcg4KgIzaAVWkFkZdGa6knQMiRC+gJ9Rfgfs7ovMK3GtA6UGW6uI0tL8PJHP0GOv2wechULMY06TIW2+5v4Vf3qx3EGbdonGY5OBRoh9rtX/6RmuRJJRRKzC4COK4uFcjWUYIkrhmhc1eltRoad3rFrpQ8AaBu+QpjxeMbrvHD4YRYtarfRwWm/m+a//zfAKlxA2VPFHt5GVippfBhF0RuRMEq8gDGNeaA4aEtyzIuCDaGSWDHllIsTajx+YEkx4upEuNx+CSkcdp8dLxEG+S/TPvkuJTxm5VBiV03paX+ZMNY19FZd4m9QX9yy++Mu1Vs7X949XH1jGA/6Kwf1xvQVTeMuYsrb8Vv40zYRdcPF9cVnQmFQ=='}}]


In [29]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="The Academic Regulations explain university rules, grading, attendance, and academic policies.",
        metadata={"source": "Academic Regulations"}
    ),
    Document(
        page_content="Course Information contains details about modules, prerequisites, and assessments.",
        metadata={"source": "Course Information"}
    ),
    Document(
        page_content="Campus Services include the library, IT support, careers service, and student wellbeing.",
        metadata={"source": "Campus Services"}
    ),
]

print(f"Loaded {len(documents)} documents.")

for doc in documents:
    print(f"- {doc.metadata['source']}")

Loaded 3 documents.
- Academic Regulations
- Course Information
- Campus Services


In [30]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

print("Vector database created successfully!")

Vector database created successfully!


In [31]:
retriever = vectorstore.as_retriever()

print("Retriever is ready!")

Retriever is ready!


In [32]:
from langchain.tools import tool

@tool
def search_university_knowledge(question: str) -> str:
    """
    Search the university knowledge base for information about
    academic regulations, courses, assessments, and campus services.
    """
    docs = retriever.invoke(question)

    if not docs:
        return "No relevant university information was found."

    results = []

    for doc in docs:
        source = doc.metadata.get("source", "Unknown source")
        results.append(
            f"Source: {source}\n"
            f"Information: {doc.page_content}"
        )

    return "\n\n".join(results)

print("University search tool is ready!")

University search tool is ready!


In [33]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[search_university_knowledge],
    system_prompt="""
You are UniGuide AI, a university knowledge assistant.

Always use the search_university_knowledge tool before answering
questions about academic regulations, courses, assessments,
campus services, the library, IT support, or student wellbeing.

Answer only from the information returned by the tool.
Mention the source in your answer.
"""
)

print("Agent is ready!")

Agent is ready!


In [34]:
agent_result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What campus services are available?"
            }
        ]
    }
)

print(agent_result["messages"][-1].content)

[{'type': 'text', 'text': 'According to the **Campus Services** source, the available campus services include:\n\n* The library\n* IT support\n* Careers service\n* Student wellbeing', 'extras': {'signature': 'EtMDCtADARFNMg8VOEhYGIKV8ABpuXsQ91Api3To4FsEZtm+bxW3Xa6CTvuAs67M2SK+6lRCylFAUcoEg8pzl7iADoISmM3v+XdI7ni/wkN7tdXg3TNYJ1GtUkoPXTq1SgAOWijGlfjl2F3VZ5MZ9Xg3l6pmuQ2+7S1sXYJfaRuUnuc0oxTrQZpDeyBaQ4I4DI4vRGzFGyYOQF3yuyWKs49QPRtulNYSulWJhJnT0P7V2QKlDNaMMJD6+GaZ3Rk1yXKdu38NUwMPfMAuqeBGZWRfhObfYv5PBjhBeEWTUuZcCFV98dIzGmEQ3gy29SpL4PXqDDb5lGEnRY+4Y9rzoymh8fluS8EvBERKqGr/NfUEDqHBgmaLqlbrkfAsfRix7qySdXv/oAz4Kcm1sU9VNepbTmn3O04SCvkQ9dRmdLgxeKG6hK/1VC8BY5xqAPRUs47RhGuDrUuXaofW+AF+TEO/DmqcXL/joI5osz+oeOhbSNfl8rZA9ENgi1jNcvveQMM1kaNuiCmXZ8jqKWkqnDf4xYECQwbLY9pFcnhYyha/VMEtaYVE4mxFXJBn2n0yIGJo9Ez656ZZv8lwHDQxr1oi0JbfBjGEU+NR60EAN2o48YAM4HE='}}]


In [35]:
query = "What campus services are available?"

results = retriever.invoke(query)

for doc in results:
    print(doc.page_content)
    print("Source:", doc.metadata["source"])
    print("-" * 50)

Campus Services include the library, IT support, careers service, and student wellbeing.
Source: Campus Services
--------------------------------------------------
Campus Services include the library, IT support, careers service, and student wellbeing.
Source: Campus Services
--------------------------------------------------
Campus Services include the library, IT support, careers service, and student wellbeing.
Source: Campus Services
--------------------------------------------------
The Academic Regulations explain university rules, grading, attendance, and academic policies.
Source: Academic Regulations
--------------------------------------------------


In [36]:
query = "What campus services are available?"

docs = retriever.invoke(query)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
You are UniGuide AI.

Answer the user's question using ONLY the information below.

Context:
{context}

Question:
{query}
"""

response = llm.invoke(prompt)

print(response.content)

[{'type': 'text', 'text': 'Based on the information provided, the campus services available are:\n* The library\n* IT support\n* Careers service\n* Student wellbeing', 'extras': {'signature': 'EpYHCpMHARFNMg8naP7+kE24FJ5feymGX2QY/ngIYrPRuMa7vNblpI6SXcUtvwHFuVIYroadXMoi/ftW64AK8IzFiKmKZA2Ae38N9hcqpS+LRWhWCTy6WSgAmq6TEy27DdhpkY7T3opZinFcn+pukysGr3g3X/+QgrZ+Z76kXyBCLRvfRaHntSafdmedct2f2LWvpS4WO/n5Grlv1m5tTWmhvajnNavrsql4Io+iphXwmcsE9QtEDnrOFFoMiD5bpIEH/lkTg/fbVqQTfDFnPzbeiv4EzpolRLIbLAJma/Rlqs9nPS0giNtZqKYOF1anrJKKfUgUPFLJRFaF6V/YJ+U86/BGFU9QeCldouvGFcVcmvcLd5szfpaQ7HFQEn2mKx5cGSckzK13inYLTlCkEli2KCARXsLolvU4/S7Hsd2fhn7PhnZz7urf+vfAZ7WlcxEa3W2lR8WML1xjHmqyzhcu2pjjPpfeqDqtAXPMj0jJvcPsNxNlIkx4Y0Mx4BtcXfV2p3wmmMQOKKdB8Dlz5+VHnAQfodQxwSzrw8XMA4Rv+lmwnmRDzM9BP1Jjl2TFpKvpZ+qQfmTgVVYIoHtQ29iZ8oivuvXjUnboUqS2uw25WBvs47wqNPsky6/ZzByGUAMSkAjbz0PQYPUn2/o3/7LNcdBzbYsh7/o5EXwPew/SlQkobxm9mcyBLjm2utA8gYdYCYPmFPT7NytPeC7NvJF9ZV5TeiNBsqj5NSnAWXL3aUSHiMufOBlwtZU0l8Pvc5/HO1AIXBpN0hEA1UzyHpBrHk7gzq7CX0XZjle

In [37]:
from typing_extensions import TypedDict


class UniGuideState(TypedDict):
    question: str
    category: str
    answer: str
    approved: bool


def router(state: UniGuideState):
    question = state["question"].lower()

    if any(word in question for word in ["grade", "attendance", "regulation", "rules"]):
        category = "academic_regulations"

    elif any(word in question for word in ["course", "module", "assessment", "prerequisite"]):
        category = "course_information"

    else:
        category = "campus_services"

    print("Selected category:", category)

    return {"category": category}

In [38]:
import time

def rag_node(state: UniGuideState):
    question = state["question"]

    docs = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
You are UniGuide AI.

Answer the user's question using ONLY the information below.

Context:
{context}

Question:
{question}
"""

    max_attempts = 5

    for attempt in range(max_attempts):
        try:
            print(f"Gemini attempt: {attempt + 1}")

            response = llm.invoke(prompt)

            if isinstance(response.content, list):
                answer = response.content[0]["text"]
            else:
                answer = response.content

            return {"answer": answer}

        except Exception as error:
            if attempt < max_attempts - 1:
                wait_time = (attempt + 1) * 10

                print(
                    f"Gemini is busy. Retrying in {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:
                return {
                    "answer": (
                        "Gemini is temporarily unavailable because of high "
                        "demand. Please try again later."
                    )
                }

In [39]:
from langgraph.types import interrupt

def human_review_node(state: UniGuideState):
    decision = interrupt(
        {
            "question": state["question"],
            "answer": state["answer"],
            "message": "Review the answer. Type approve or reject."
        }
    )

    if decision.lower() == "approve":
        return {"approved": True}

    return {
        "approved": False,
        "answer": "The answer was rejected by the human reviewer."
    }

In [40]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

graph = StateGraph(UniGuideState)

# Nodes
graph.add_node("router", router)
graph.add_node("rag", rag_node)
graph.add_node("human_review", human_review_node)

# Flow
graph.add_edge(START, "router")
graph.add_edge("router", "rag")
graph.add_edge("rag", "human_review")
graph.add_edge("human_review", END)

# Memory
memory = MemorySaver()

# Compile
app = graph.compile(checkpointer=memory)

print("LangGraph with Human-in-the-Loop is ready!")

LangGraph with Human-in-the-Loop is ready!


In [41]:
config = {
    "configurable": {
        "thread_id": "user-1"
    }
}

result = app.invoke(
    {
        "question": "What is the attendance policy?"
    },
    config=config
)

print(result)

Selected category: academic_regulations
Gemini attempt: 1
{'question': 'What is the attendance policy?', 'category': 'academic_regulations', 'answer': 'Based on the provided information, the specific details of the attendance policy are not mentioned. However, the Academic Regulations explain attendance.', '__interrupt__': [Interrupt(value={'question': 'What is the attendance policy?', 'answer': 'Based on the provided information, the specific details of the attendance policy are not mentioned. However, the Academic Regulations explain attendance.', 'message': 'Review the answer. Type approve or reject.'}, id='4ab0a3c596d2454efd71e22e4f9d7b6a')]}


In [42]:
from langgraph.types import Command

decision = input("Type approve or reject: ")

result = app.invoke(
    Command(resume=decision),
    config=config
)

print(result)

Type approve or reject: approve
{'question': 'What is the attendance policy?', 'category': 'academic_regulations', 'answer': 'Based on the provided information, the specific details of the attendance policy are not mentioned. However, the Academic Regulations explain attendance.', 'approved': True}


In [43]:
config = {
    "configurable": {
        "thread_id": "user-1"
    }
}

result = app.invoke(
    {
        "question": "What is the attendance policy?"
    },
    config=config
)

print("Category:", result["category"])
print("Answer:", result["answer"])

Selected category: academic_regulations
Gemini attempt: 1
Category: academic_regulations
Answer: Based on the provided information, the specific details of the attendance policy are not stated. However, the attendance policy is explained in the Academic Regulations.
